# 📈 Eco-Logic RL Training Analysis
Analyse Q-Learning episode history: rewards, PUE progression, thermal safety, GPU density.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import json
from pathlib import Path

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✓')

In [ ]:
# Load simulation results
results_path = Path('../data/rl_simulation_results.json')
with open(results_path) as f:
    data = json.load(f)

summary = data['summary']
df = pd.DataFrame(data['episodes'])

print('=== Training Summary ===')
for k, v in summary.items():
    print(f'  {k:<30}: {v}')
print(f'\nDataFrame shape: {df.shape}')
df.head()

In [ ]:
# Rolling averages
df['reward_ma10']  = df['reward'].rolling(10).mean()
df['pue_ma10']     = df['pue'].rolling(10).mean()
df['temp_ma10']    = df['avg_temp_c'].rolling(10).mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Eco-Logic RL Training Analysis (200 Episodes)', fontsize=14, fontweight='bold')

# 1. Reward curve
ax = axes[0,0]
ax.plot(df['episode'], df['reward'], alpha=0.3, color='#39ff14', linewidth=0.8, label='Episode reward')
ax.plot(df['episode'], df['reward_ma10'], color='#39ff14', linewidth=2, label='MA-10')
ax.set_title('RL Reward Convergence'); ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward')
ax.legend(); ax.axhline(df['reward'].max(), color='white', linestyle='--', alpha=0.4, label='Best')

# 2. PUE progression
ax = axes[0,1]
ax.plot(df['episode'], df['pue'], alpha=0.3, color='#00e5ff', linewidth=0.8)
ax.plot(df['episode'], df['pue_ma10'], color='#00e5ff', linewidth=2, label='PUE MA-10')
ax.axhline(1.20, color='#ff6b35', linestyle='--', linewidth=1.5, label='Target PUE 1.20')
ax.axhline(1.61, color='gray', linestyle=':', linewidth=1, label='Baseline 1.61')
ax.set_title('PUE Optimisation'); ax.set_xlabel('Episode'); ax.set_ylabel('PUE')
ax.legend(); ax.set_ylim(1.1, 1.75)

# 3. Temperature trend
ax = axes[1,0]
ax.plot(df['episode'], df['avg_temp_c'], alpha=0.3, color='#ff6b35', linewidth=0.8)
ax.plot(df['episode'], df['temp_ma10'], color='#ff6b35', linewidth=2, label='Avg Temp MA-10')
ax.axhline(78, color='#ff2d2d', linestyle='--', linewidth=1.5, label='Spike threshold 78°C')
ax.axhline(85, color='#aa0000', linestyle='--', linewidth=1.5, label='Critical 85°C')
ax.set_title('Thermal Safety'); ax.set_xlabel('Episode'); ax.set_ylabel('Avg Temp (°C)')
ax.legend()

# 4. GPU density + deals
ax = axes[1,1]
ax2 = ax.twinx()
ax.plot(df['episode'], df['gpu_density_x'], color='#b44dff', linewidth=2, label='GPU Density (×)')
ax2.plot(df['episode'], df['deals_unlocked_M'], color='#ffb800', linewidth=2, linestyle='--', label='Deals ($M)')
ax.axhline(4.0, color='#b44dff', linestyle=':', alpha=0.5)
ax2.axhline(50, color='#ffb800', linestyle=':', alpha=0.5)
ax.set_title('Business Impact'); ax.set_xlabel('Episode')
ax.set_ylabel('GPU Density (×)', color='#b44dff')
ax2.set_ylabel('Deals Unblocked ($M)', color='#ffb800')
ax.legend(loc='upper left'); ax2.legend(loc='center left')

plt.tight_layout()
plt.savefig('../data/rl_training_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved → data/rl_training_analysis.png')

In [ ]:
# Reward distribution analysis
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Reward & PUE Distribution Analysis', fontsize=13, fontweight='bold')

# Histogram
axes[0].hist(df['reward'], bins=30, color='#39ff14', alpha=0.7, edgecolor='black')
axes[0].axvline(df['reward'].mean(), color='red', linestyle='--', label=f'Mean={df["reward"].mean():.1f}')
axes[0].set_title('Reward Distribution'); axes[0].legend()

# PUE histogram
axes[1].hist(df['pue'], bins=30, color='#00e5ff', alpha=0.7, edgecolor='black')
axes[1].axvline(1.20, color='#ff6b35', linestyle='--', linewidth=2, label='Target PUE')
axes[1].set_title('PUE Distribution'); axes[1].legend()

# Scatter: reward vs PUE
sc = axes[2].scatter(df['pue'], df['reward'], c=df['episode'], cmap='plasma', alpha=0.6, s=20)
plt.colorbar(sc, ax=axes[2], label='Episode')
axes[2].set_xlabel('PUE'); axes[2].set_ylabel('Reward')
axes[2].set_title('Reward vs PUE (coloured by episode)')

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
stats = df[['reward','pue','avg_temp_c','gpu_density_x','deals_unlocked_M']].describe()
print(stats.to_string())

pue_reduction = (df['pue'].iloc[0] - df['pue'].iloc[-1]) / df['pue'].iloc[0] * 100
print(f'\nPUE reduction over training : {pue_reduction:.1f}%')
print(f'GPU density achieved        : {df["gpu_density_x"].iloc[-1]:.2f}×')
print(f'Deals unlocked              : ${df["deals_unlocked_M"].iloc[-1]:.1f}M')
print(f'Digital twin accuracy       : {df["twin_accuracy_pct"].iloc[-1]:.1f}%')